In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, lit, desc, hour, dayofmonth, dayofweek
from pyspark.sql.functions import split, regexp_extract
from pyspark.sql.types import DoubleType, FloatType

spark = SparkSession.builder \
    .appName("Analisis Ecommerce Octubre") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()

spark

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
file_path = "/content/drive/MyDrive/Big Data/2019-Oct.csv"

df = (spark.read.option("header", True)
          .option("inferSchema", True)
          .csv(file_path))
df.show(5, truncate=False)

+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|2019-10-01 00:00:00|view      |44600062  |2103807459595387724|NULL                               |shiseido|35.79  |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-10-01 00:00:00|view      |3900821   |2053013552326770905|appliances.environment.water_heater|aqua    |33.2   |554748717|9333dfbd-b87a-4708-9857-6336556b0fcc|
|2019-10-01 00:00:01|view      |17200506  |2053013559792632471|furniture.living_room.sofa         |NULL    |543.1  |519107250|566511c2-e2e3-422b-b695-cf8e6e792ca8|
|2019-10-01 00:0

In [11]:
df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



## Estadísticas

In [12]:
#Numero de registros y columnas
num_rows = df.count()
num_cols = len(df.columns)

print(f"Número de registros: {num_rows:,}")
print(f"Número de columnas: {num_cols}")

Número de registros: 42,448,764
Número de columnas: 9


In [ ]:
#Estadísticas de variable precio
df.select("price").describe().show()
price_percentiles = df.approxQuantile("price", [0.25, 0.5, 0.75], 0.01)

print("Percentil 25:", price_percentiles[0])
print("Mediana:", price_percentiles[1])
print("Percentil 75:", price_percentiles[2])

+-------+-----------------+
|summary|            price|
+-------+-----------------+
|  count|         42448764|
|   mean|290.3236606848809|
| stddev|358.2691553394021|
|    min|              0.0|
|    max|          2574.07|
+-------+-----------------+

Percentil 25: 64.78
Mediana: 161.83
Percentil 75: 357.8


In [ ]:
#Frecuencia de event_type
event_type_counts = df.groupBy("event_type") \
                      .count() \
                      .withColumn("percentage", (col("count") / lit(num_rows)) * 100) \
                      .orderBy(desc("count"))

event_type_counts.show(truncate=False)

+----------+--------+------------------+
|event_type|count   |percentage        |
+----------+--------+------------------+
|view      |40779399|96.06734132470854 |
|cart      |926516  |2.182668970055288 |
|purchase  |742849  |1.7499897052361761|
+----------+--------+------------------+



In [ ]:
#Valores faltantes por columna
from pyspark.sql.functions import col, count, when, isnan
import pandas as pd

# Número total de registros
num_rows = df.count()

# Crear expresiones para contar valores faltantes
missing_exprs = []

for c, dtype in df.dtypes:
    if dtype in ["double", "float"]:
        missing_exprs.append(
            count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
        )
    else:
        missing_exprs.append(
            count(when(col(c).isNull(), c)).alias(c)
        )

# Ejecutar conteo de nulos
missing_result = df.select(missing_exprs).collect()[0].asDict()

# Convertir a Pandas para evitar error de Python worker en Spark
missing_summary_pd = pd.DataFrame(
    list(missing_result.items()),
    columns=["variable", "missing_values"]
)

missing_summary_pd["missing_percentage"] = (
    missing_summary_pd["missing_values"] / num_rows
) * 100

missing_summary_pd = missing_summary_pd.sort_values(
    by="missing_percentage",
    ascending=False
)

missing_summary_pd

,variable,missing_values,missing_percentage
4,category_code,13515609,31.839818
5,brand,6113008,14.400909
8,user_session,2,0.000005
0,event_time,0,0.000000
1,event_type,0,0.000000
3,category_id,0,0.000000
2,product_id,0,0.000000
6,price,0,0.000000
7,user_id,0,0.000000


In [ ]:
#Valores únicos por variable
columns_to_check = ["product_id", "category_id", "category_code", "brand", "user_id", "user_session"]

for c in columns_to_check:
    print(f"{c}: {df.select(c).distinct().count():,} valores únicos")

product_id: 166,794 valores únicos
category_id: 624 valores únicos
category_code: 127 valores únicos
brand: 3,446 valores únicos
user_id: 3,022,290 valores únicos
user_session: 9,244,422 valores únicos


In [ ]:
#Categorías más frecuentes
df.groupBy("category_code") \
  .count() \
  .withColumn("percentage", (col("count") / lit(num_rows)) * 100) \
  .orderBy(desc("count")) \
  .show(20, truncate=False)

+--------------------------------+--------+------------------+
|category_code                   |count   |percentage        |
+--------------------------------+--------+------------------+
|NULL                            |13515609|31.839817526842477|
|electronics.smartphone          |11507231|27.108518401148263|
|electronics.clocks              |1311033 |3.088506887974406 |
|computers.notebook              |1137623 |2.6799908708767117|
|electronics.video.tv            |1113750 |2.623751306398462 |
|electronics.audio.headphone     |1100188 |2.591802201826183 |
|appliances.kitchen.refrigerators|887755  |2.0913565351396333|
|appliances.kitchen.washer       |869404  |2.04812559442249  |
|appliances.environment.vacuum   |801670  |1.888559110931946 |
|apparel.shoes                   |763901  |1.7995836109621473|
|auto.accessories.player         |470208  |1.1077071643358096|
|computers.desktop               |424242  |0.9994213259071572|
|apparel.shoes.keds              |410304  |0.9665864476

In [ ]:
#Marcas más frecuentes
df.groupBy("brand") \
  .count() \
  .withColumn("percentage", (col("count") / lit(num_rows)) * 100) \
  .orderBy(desc("count")) \
  .show(20, truncate=False)

+--------+-------+------------------+
|brand   |count  |percentage        |
+--------+-------+------------------+
|NULL    |6113008|14.400909293848931|
|samsung |5282775|12.44506200463222 |
|apple   |4122554|9.711835190301418 |
|xiaomi  |3083763|7.264670886530407 |
|huawei  |1111205|2.617755843256119 |
|lucente |655861 |1.5450650106090251|
|lg      |562404 |1.3249007674287054|
|bosch   |557090 |1.3123821461562462|
|oppo    |482887 |1.1375761141125333|
|sony    |456644 |1.0757533482011397|
|acer    |428153 |1.008634786162443 |
|cordiant|368277 |0.8675800313055051|
|artel   |338203 |0.7967322676344593|
|lenovo  |338103 |0.7964966895149174|
|hp      |305242 |0.7190833636522372|
|indesit |290101 |0.6834144805723907|
|dauscher|289853 |0.6828302468359266|
|philips |282699 |0.6659769881638956|
|redmond |279734 |0.6589920969194769|
|respect |271167 |0.638810119418318 |
+--------+-------+------------------+
only showing top 20 rows


## Particionamiento

In [ ]:
from pyspark.sql.functions import (
    col, when, hour, split, lit, desc, count, coalesce
)

In [ ]:
df_part = df

In [ ]:
df_part = df_part.withColumn(
    "category_group",
    when(
        col("category_code").isNull(),
        "unknown"
    ).otherwise(
        split(col("category_code"), "\\.").getItem(0)
    )
)

df_part.select("category_code", "category_group").show(10, truncate=False)

+-----------------------------------+--------------+
|category_code                      |category_group|
+-----------------------------------+--------------+
|NULL                               |unknown       |
|appliances.environment.water_heater|appliances    |
|furniture.living_room.sofa         |furniture     |
|computers.notebook                 |computers     |
|electronics.smartphone             |electronics   |
|computers.desktop                  |computers     |
|NULL                               |unknown       |
|NULL                               |unknown       |
|apparel.shoes.keds                 |apparel       |
|electronics.smartphone             |electronics   |
+-----------------------------------+--------------+
only showing top 10 rows


In [ ]:
df_part = df_part.withColumn(
    "event_hour",
    hour(col("event_time"))
)

df_part = df_part.withColumn(
    "time_period",
    when((col("event_hour") >= 0) & (col("event_hour") <= 5), "night")
    .when((col("event_hour") >= 6) & (col("event_hour") <= 11), "morning")
    .when((col("event_hour") >= 12) & (col("event_hour") <= 17), "afternoon")
    .when((col("event_hour") >= 18) & (col("event_hour") <= 23), "evening")
    .otherwise("unknown")
)

df_part.select("event_time", "event_hour", "time_period").show(10, truncate=False)

+-------------------+----------+-----------+
|event_time         |event_hour|time_period|
+-------------------+----------+-----------+
|2019-10-01 00:00:00|0         |night      |
|2019-10-01 00:00:00|0         |night      |
|2019-10-01 00:00:01|0         |night      |
|2019-10-01 00:00:01|0         |night      |
|2019-10-01 00:00:04|0         |night      |
|2019-10-01 00:00:05|0         |night      |
|2019-10-01 00:00:08|0         |night      |
|2019-10-01 00:00:08|0         |night      |
|2019-10-01 00:00:10|0         |night      |
|2019-10-01 00:00:11|0         |night      |
+-------------------+----------+-----------+
only showing top 10 rows


In [ ]:
p25 = 64.78
p75 = 357.77

df_part = df_part.withColumn(
    "price_range",
    when(col("price") <= p25, "low")
    .when((col("price") > p25) & (col("price") <= p75), "medium")
    .when(col("price") > p75, "high")
    .otherwise("unknown")
)

df_part.select("price", "price_range").show(10, truncate=False)

+-------+-----------+
|price  |price_range|
+-------+-----------+
|35.79  |low        |
|33.2   |low        |
|543.1  |high       |
|251.74 |medium     |
|1081.98|high       |
|908.62 |high       |
|380.96 |high       |
|41.16  |low        |
|102.71 |medium     |
|566.01 |high       |
+-------+-----------+
only showing top 10 rows


In [ ]:
df_part.select(
    "event_type",
    "category_code",
    "category_group",
    "price",
    "price_range",
    "event_time",
    "event_hour",
    "time_period"
).show(20, truncate=False)

+----------+-----------------------------------+--------------+-------+-----------+-------------------+----------+-----------+
|event_type|category_code                      |category_group|price  |price_range|event_time         |event_hour|time_period|
+----------+-----------------------------------+--------------+-------+-----------+-------------------+----------+-----------+
|view      |NULL                               |unknown       |35.79  |low        |2019-10-01 00:00:00|0         |night      |
|view      |appliances.environment.water_heater|appliances    |33.2   |low        |2019-10-01 00:00:00|0         |night      |
|view      |furniture.living_room.sofa         |furniture     |543.1  |high       |2019-10-01 00:00:01|0         |night      |
|view      |computers.notebook                 |computers     |251.74 |medium     |2019-10-01 00:00:01|0         |night      |
|view      |electronics.smartphone             |electronics   |1081.98|high       |2019-10-01 00:00:04|0       

In [ ]:
df_part.groupBy("event_type") \
    .count() \
    .withColumn("probability", col("count") / lit(num_rows)) \
    .withColumn("percentage", col("probability") * 100) \
    .orderBy(desc("count")) \
    .show(truncate=False)

+----------+--------+--------------------+------------------+
|event_type|count   |probability         |percentage        |
+----------+--------+--------------------+------------------+
|view      |40779399|0.9606734132470853  |96.06734132470854 |
|cart      |926516  |0.021826689700552883|2.182668970055288 |
|purchase  |742849  |0.01749989705236176 |1.7499897052361761|
+----------+--------+--------------------+------------------+



In [ ]:
df_part.groupBy("category_group") \
    .count() \
    .withColumn("probability", col("count") / lit(num_rows)) \
    .withColumn("percentage", col("probability") * 100) \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

+--------------+--------+---------------------+--------------------+
|category_group|count   |probability          |percentage          |
+--------------+--------+---------------------+--------------------+
|electronics   |16135623|0.3801199723977829   |38.01199723977829   |
|unknown       |13515609|0.3183981752684248   |31.839817526842477  |
|appliances    |4967294 |0.11701857797320082  |11.701857797320082  |
|computers     |2324217 |0.05475346702674311  |5.475346702674311   |
|apparel       |1542924 |0.03634791345161428  |3.634791345161428   |
|furniture     |1247160 |0.029380360756793768 |2.938036075679377   |
|auto          |1013115 |0.02386677265797421  |2.386677265797421   |
|construction  |730834  |0.01721684994173211  |1.7216849941732109  |
|kids          |520619  |0.012264644501781018 |1.2264644501781017  |
|accessories   |238238  |0.005612366004343495 |0.5612366004343495  |
|sport         |176616  |0.0041606865161020945|0.41606865161020945 |
|medicine      |14806   |3.4879696

In [ ]:
df_part.groupBy("price_range") \
    .count() \
    .withColumn("probability", col("count") / lit(num_rows)) \
    .withColumn("percentage", col("probability") * 100) \
    .orderBy(desc("count")) \
    .show(truncate=False)

+-----------+--------+-------------------+------------------+
|price_range|count   |probability        |percentage        |
+-----------+--------+-------------------+------------------+
|medium     |21234978|0.5002496185754667 |50.02496185754667 |
|high       |10700699|0.2520850548204419 |25.208505482044192|
|low        |10513087|0.24766532660409146|24.766532660409148|
+-----------+--------+-------------------+------------------+



In [ ]:
df_part.groupBy("time_period") \
    .count() \
    .withColumn("probability", col("count") / lit(num_rows)) \
    .withColumn("percentage", col("probability") * 100) \
    .orderBy(desc("count")) \
    .show(truncate=False)

+-----------+--------+-------------------+------------------+
|time_period|count   |probability        |percentage        |
+-----------+--------+-------------------+------------------+
|afternoon  |15940761|0.37552945004476457|37.552945004476456|
|morning    |13837692|0.3259857460160677 |32.59857460160677 |
|night      |7526440 |0.177306458204531  |17.730645820453102|
|evening    |5143871 |0.1211783457346367 |12.117834573463671|
+-----------+--------+-------------------+------------------+



In [ ]:
partition_combinations = df_part.groupBy(
    "event_type",
    "category_group",
    "price_range",
    "time_period"
).count()

partition_combinations = partition_combinations.withColumn(
    "probability",
    col("count") / lit(num_rows)
).withColumn(
    "percentage",
    col("probability") * 100
)

partition_combinations.orderBy(desc("count")).show(50, truncate=False)

+----------+--------------+-----------+-----------+-------+---------------------+-------------------+
|event_type|category_group|price_range|time_period|count  |probability          |percentage         |
+----------+--------------+-----------+-----------+-------+---------------------+-------------------+
|view      |electronics   |medium     |afternoon  |3060638|0.0721019344638633   |7.21019344638633   |
|view      |electronics   |medium     |morning    |2595235|0.061138058106945115 |6.113805810694512  |
|view      |electronics   |high       |afternoon  |2207636|0.05200707375131111  |5.2007073751311115 |
|view      |unknown       |medium     |afternoon  |2199336|0.05181154391209129  |5.181154391209129  |
|view      |unknown       |low        |afternoon  |2067948|0.048716330115053524 |4.871633011505352  |
|view      |electronics   |high       |morning    |1882043|0.044336815083708916 |4.433681508370892  |
|view      |unknown       |medium     |morning    |1833877|0.04320212951312316  |4

In [ ]:
# Guardar las principales combinaciones de particionamiento
top_partition_combinations = partition_combinations.orderBy(desc("count")).limit(20)

top_partition_combinations_pd = top_partition_combinations.toPandas()
top_partition_combinations_pd

,event_type,category_group,price_range,time_period,count,probability,percentage
0,view,electronics,medium,afternoon,3060638,0.072102,7.210193
1,view,electronics,medium,morning,2595235,0.061138,6.113806
2,view,electronics,high,afternoon,2207636,0.052007,5.200707
3,view,unknown,medium,afternoon,2199336,0.051812,5.181154
4,view,unknown,low,afternoon,2067948,0.048716,4.871633
5,view,electronics,high,morning,1882043,0.044337,4.433682
6,view,unknown,medium,morning,1833877,0.043202,4.320213
7,view,unknown,low,morning,1801412,0.042437,4.243733
8,view,electronics,medium,night,1403775,0.033070,3.306987
9,view,unknown,medium,night,1050138,0.024739,2.473895


## Extracción de muestras de ejemplo por regla de particionamiento

En esta sección se muestran ejemplos de extracción de registros que cumplen con reglas específicas de particionamiento. Cada regla se define por la combinación de cuatro variables: `event_type`, `category_group`, `price_range` y `time_period`.

El objetivo de esta etapa no es construir todavía la muestra final de entrenamiento o prueba, sino comprobar que el código permite recuperar correctamente subconjuntos de la base de datos original D de acuerdo con las reglas definidas en la etapa de particionamiento.

In [ ]:
def extraer_muestra_particion(event_type, category_group, price_range, time_period, n=10):
    """
    Filtra la base particionada usando una regla de particionamiento
    y muestra una muestra de ejemplo de n registros.
    """

    particion = df_part.filter(
        (col("event_type") == event_type) &
        (col("category_group") == category_group) &
        (col("price_range") == price_range) &
        (col("time_period") == time_period)
    )

    total_registros = particion.count()

    print("Regla de particionamiento:")
    print(f"event_type = {event_type}")
    print(f"category_group = {category_group}")
    print(f"price_range = {price_range}")
    print(f"time_period = {time_period}")
    print(f"Total de registros en la partición: {total_registros:,}")

    particion.show(n, truncate=False)

    return particion

In [ ]:
#Ejemplo 1
muestra_p1 = extraer_muestra_particion(
    event_type="view",
    category_group="electronics",
    price_range="medium",
    time_period="morning",
    n=10
)

Regla de particionamiento:
event_type = view
category_group = electronics
price_range = medium
time_period = morning
Total de registros en la partición: 2,595,235
+-------------------+----------+----------+-------------------+----------------------+-------+------+---------+------------------------------------+--------------+----------+-----------+-----------+
|event_time         |event_type|product_id|category_id        |category_code         |brand  |price |user_id  |user_session                        |category_group|event_hour|time_period|price_range|
+-------------------+----------+----------+-------------------+----------------------+-------+------+---------+------------------------------------+--------------+----------+-----------+-----------+
|2019-10-01 06:00:00|view      |1004768   |2053013555631882655|electronics.smartphone|samsung|253.31|551086352|ea5e607f-86b0-4f82-a982-53fcbee2408f|electronics   |6         |morning    |medium     |
|2019-10-01 06:00:00|view      |1005153  

In [ ]:
#Ejemplo 2
muestra_p2 = extraer_muestra_particion(
    event_type="view",
    category_group="electronics",
    price_range="medium",
    time_period="night",
    n=10
)

Regla de particionamiento:
event_type = view
category_group = electronics
price_range = medium
time_period = night
Total de registros en la partición: 1,403,775
+-------------------+----------+----------+-------------------+---------------------------+-------+------+---------+------------------------------------+--------------+----------+-----------+-----------+
|event_time         |event_type|product_id|category_id        |category_code              |brand  |price |user_id  |user_session                        |category_group|event_hour|time_period|price_range|
+-------------------+----------+----------+-------------------+---------------------------+-------+------+---------+------------------------------------+--------------+----------+-----------+-----------+
|2019-10-01 00:00:18|view      |1801995   |2053013554415534427|electronics.video.tv       |haier  |193.03|537192226|e3151795-c355-4efa-acf6-e1fe1bebeee5|electronics   |0         |night      |medium     |
|2019-10-01 00:00:23|vi

In [ ]:
#Ejemplo 3
muestra_p3 = extraer_muestra_particion(
    event_type="view",
    category_group="electronics",
    price_range="high",
    time_period="morning",
    n=10
)

Regla de particionamiento:
event_type = view
category_group = electronics
price_range = high
time_period = morning
Total de registros en la partición: 1,882,043
+-------------------+----------+----------+-------------------+----------------------+-------+-------+---------+------------------------------------+--------------+----------+-----------+-----------+
|event_time         |event_type|product_id|category_id        |category_code         |brand  |price  |user_id  |user_session                        |category_group|event_hour|time_period|price_range|
+-------------------+----------+----------+-------------------+----------------------+-------+-------+---------+------------------------------------+--------------+----------+-----------+-----------+
|2019-10-01 06:00:01|view      |1004873   |2053013555631882655|electronics.smartphone|samsung|388.81 |518566925|6a7a239d-36de-432a-a59a-8beec5a2f9b9|electronics   |6         |morning    |high       |
|2019-10-01 06:00:02|view      |1801712

In [ ]:
#Ejemplo 4
muestra_purchase = extraer_muestra_particion(
    event_type="purchase",
    category_group="electronics",
    price_range="medium",
    time_period="morning",
    n=10
)

Regla de particionamiento:
event_type = purchase
category_group = electronics
price_range = medium
time_period = morning
Total de registros en la partición: 103,841
+-------------------+----------+----------+-------------------+---------------------------+-------+------+---------+------------------------------------+--------------+----------+-----------+-----------+
|event_time         |event_type|product_id|category_id        |category_code              |brand  |price |user_id  |user_session                        |category_group|event_hour|time_period|price_range|
+-------------------+----------+----------+-------------------+---------------------------+-------+------+---------+------------------------------------+--------------+----------+-----------+-----------+
|2019-10-01 06:00:11|purchase  |1801766   |2053013554415534427|electronics.video.tv       |artel  |150.7 |515807042|d93a48cc-8b48-40b4-99dc-be5af45166cf|electronics   |6         |morning    |medium     |
|2019-10-01 06:00:1

In [ ]:
#Extraer las 10 reglas de particionamiento más frecuentes
top_10_rules = partition_combinations.orderBy(desc("count")).limit(10).collect()

for i, rule in enumerate(top_10_rules, start=1):
    print("=" * 80)
    print(f"Muestra de ejemplo para la partición P{i}")

    muestra = df_part.filter(
        (col("event_type") == rule["event_type"]) &
        (col("category_group") == rule["category_group"]) &
        (col("price_range") == rule["price_range"]) &
        (col("time_period") == rule["time_period"])
    )

    print(f"event_type: {rule['event_type']}")
    print(f"category_group: {rule['category_group']}")
    print(f"price_range: {rule['price_range']}")
    print(f"time_period: {rule['time_period']}")
    print(f"Registros en la partición: {rule['count']:,}")
    print(f"Probabilidad: {rule['probability']:.4f}")
    print(f"Porcentaje: {rule['percentage']:.2f}%")

    muestra.show(5, truncate=False)

Muestra de ejemplo para la partición P1
event_type: view
category_group: electronics
price_range: medium
time_period: afternoon
Registros en la partición: 3,060,638
Probabilidad: 0.0721
Porcentaje: 7.21%
+-------------------+----------+----------+-------------------+---------------------------+-------+------+---------+------------------------------------+--------------+----------+-----------+-----------+
|event_time         |event_type|product_id|category_id        |category_code              |brand  |price |user_id  |user_session                        |category_group|event_hour|time_period|price_range|
+-------------------+----------+----------+-------------------+---------------------------+-------+------+---------+------------------------------------+--------------+----------+-----------+-----------+
|2019-10-01 12:00:00|view      |1004863   |2053013555631882655|electronics.smartphone     |samsung|174.38|554989582|3be9234f-5119-4f9c-900e-b15db47b2321|electronics   |12        |after

In [ ]:
# Ejemplo de muestreo estratificado proporcional
# En esta etapa solo se muestra cómo se podría aplicar posteriormente.

sample_fraction = 0.001  # 0.1% de cada partición, ejemplo

sample_df = df_part.sampleBy(
    "event_type",
    fractions={
        "view": sample_fraction,
        "cart": sample_fraction,
        "purchase": sample_fraction
    },
    seed=42
)

sample_df.groupBy("event_type").count().show()

+----------+-----+
|event_type|count|
+----------+-----+
|  purchase|  783|
|      view|40927|
|      cart|  892|
+----------+-----+



In [ ]:
from pyspark.sql.functions import concat_ws

# Crear una llave única de partición
df_part = df_part.withColumn(
    "partition_key",
    concat_ws(
        "_",
        "event_type",
        "category_group",
        "price_range",
        "time_period"
    )
)

df_part.select(
    "event_type",
    "category_group",
    "price_range",
    "time_period",
    "partition_key"
).show(10, truncate=False)

+----------+--------------+-----------+-----------+---------------------------+
|event_type|category_group|price_range|time_period|partition_key              |
+----------+--------------+-----------+-----------+---------------------------+
|view      |unknown       |low        |night      |view_unknown_low_night     |
|view      |appliances    |low        |night      |view_appliances_low_night  |
|view      |furniture     |high       |night      |view_furniture_high_night  |
|view      |computers     |medium     |night      |view_computers_medium_night|
|view      |electronics   |high       |night      |view_electronics_high_night|
|view      |computers     |high       |night      |view_computers_high_night  |
|view      |unknown       |high       |night      |view_unknown_high_night    |
|view      |unknown       |low        |night      |view_unknown_low_night     |
|view      |apparel       |medium     |night      |view_apparel_medium_night  |
|view      |electronics   |high       |n

## Técnica de muestreo propuesta

Para la construcción de la muestra representativa M se propone utilizar muestreo estratificado proporcional por partición. Cada partición está definida por la combinación de `event_type`, `category_group`, `price_range` y `time_period`.

La muestra se obtendrá seleccionando registros de cada partición de acuerdo con la proporción que dicha partición representa dentro de la población original. Esto permite conservar la estructura de la base D y reducir el riesgo de sesgo, especialmente porque el dataset presenta un fuerte desbalance entre eventos de visualización, carrito y compra.